# Karachi Crime Risk — Training & Evaluation (Compare Models)

This notebook trains and compares **four** models on the leakage-safe, memory-safe features:

- Logistic Regression (baseline, sparse-friendly)
- Linear SVM (LinearSVC + calibration for probabilities)
- Random Forest (via dimensionality reduction if needed)
- XGBoost (via dimensionality reduction if needed)

## Inputs (same folder / root)
- `X_train.npz`, `X_test.npz`
- `y_train.csv`, `y_test.csv`
- `preprocessing_artifacts.joblib`

## Outputs
- Metrics table and best model selection
- `highrisk_model_bundle.joblib` (best model + preprocessing + threshold)

> Note: If the feature matrix is very wide (many one-hot columns), RandomForest/XGBoost may require dimensionality reduction.
We handle this automatically using `TruncatedSVD`.


In [1]:
# 0) Imports + Load Data
import numpy as np
import pandas as pd
from pathlib import Path

from scipy.sparse import load_npz, issparse

from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, average_precision_score,
    confusion_matrix, classification_report
)

from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.calibration import CalibratedClassifierCV
from sklearn.model_selection import StratifiedKFold
from sklearn.decomposition import TruncatedSVD

from sklearn.ensemble import RandomForestClassifier

import joblib

RANDOM_STATE = 42
DATA_DIR = Path(".")

# --- IMPORTANT: define GeoClusterAdder before joblib.load (pickle needs it) ---
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.cluster import KMeans

class GeoClusterAdder(BaseEstimator, TransformerMixin):
    def __init__(self, lat_col, lon_col, n_clusters=40, random_state=42):
        self.lat_col = lat_col
        self.lon_col = lon_col
        self.n_clusters = n_clusters
        self.random_state = random_state
        self.km_ = None

    def fit(self, X, y=None):
        X_ = X[[self.lat_col, self.lon_col]].copy()
        X_[self.lat_col] = pd.to_numeric(X_[self.lat_col], errors="coerce")
        X_[self.lon_col] = pd.to_numeric(X_[self.lon_col], errors="coerce")
        X_ = X_.fillna(X_.median(numeric_only=True))
        self.km_ = KMeans(
            n_clusters=self.n_clusters,
            random_state=self.random_state,
            n_init="auto"
        )
        self.km_.fit(X_)
        return self

    def transform(self, X):
        X_out = X.copy()
        X_ = X_out[[self.lat_col, self.lon_col]].copy()
        X_[self.lat_col] = pd.to_numeric(X_[self.lat_col], errors="coerce")
        X_[self.lon_col] = pd.to_numeric(X_[self.lon_col], errors="coerce")
        X_ = X_.fillna(X_.median(numeric_only=True))
        X_out["geo_cluster"] = self.km_.predict(X_).astype(np.int16)
        return X_out

# Load sparse matrices + targets
X_train = load_npz(DATA_DIR / "X_train.npz")
X_test  = load_npz(DATA_DIR / "X_test.npz")

y_train = pd.read_csv(DATA_DIR / "y_train.csv")["HighRisk"].astype(int).values
y_test  = pd.read_csv(DATA_DIR / "y_test.csv")["HighRisk"].astype(int).values

artifacts = joblib.load(DATA_DIR / "preprocessing_artifacts.joblib")
feature_names = artifacts.get("feature_names", None)

print("X_train:", X_train.shape, "sparse:", issparse(X_train))
print("X_test :", X_test.shape, "sparse:", issparse(X_test))
print("Train positive rate:", y_train.mean().round(4), "Test positive rate:", y_test.mean().round(4))


X_train: (80000, 158) sparse: True
X_test : (20000, 158) sparse: True
Train positive rate: 0.4628 Test positive rate: 0.4628


In [2]:
# 1) Metrics helpers

def evaluate_print(name, y_true, y_pred, y_proba=None):
    print("\n" + "="*80)
    print(name)
    print("="*80)
    print("Accuracy :", accuracy_score(y_true, y_pred))
    print("Precision:", precision_score(y_true, y_pred, zero_division=0))
    print("Recall   :", recall_score(y_true, y_pred, zero_division=0))
    print("F1       :", f1_score(y_true, y_pred, zero_division=0))
    if y_proba is not None:
        print("ROC-AUC  :", roc_auc_score(y_true, y_proba))
        print("PR-AUC   :", average_precision_score(y_true, y_proba))
    print("\nConfusion Matrix:")
    print(confusion_matrix(y_true, y_pred))
    print("\nClassification Report:")
    print(classification_report(y_true, y_pred, zero_division=0))

def metrics_dict(y_true, y_pred, y_proba=None):
    out = {
        "accuracy": accuracy_score(y_true, y_pred),
        "precision": precision_score(y_true, y_pred, zero_division=0),
        "recall": recall_score(y_true, y_pred, zero_division=0),
        "f1": f1_score(y_true, y_pred, zero_division=0),
    }
    if y_proba is not None:
        out["roc_auc"] = roc_auc_score(y_true, y_proba)
        out["pr_auc"] = average_precision_score(y_true, y_proba)
    else:
        out["roc_auc"] = np.nan
        out["pr_auc"] = np.nan
    return out

def tune_threshold_f1(y_true, y_proba):
    thresholds = np.linspace(0.05, 0.95, 19)
    best_t, best_f1 = 0.5, -1.0
    for t in thresholds:
        pred = (y_proba >= t).astype(int)
        f1 = f1_score(y_true, pred, zero_division=0)
        if f1 > best_f1:
            best_t, best_f1 = float(t), float(f1)
    return best_t, best_f1


## 2) Baseline: Dummy (most frequent)

This should be near the majority-class rate and usually has **0 recall for positives**.


In [3]:
results = []
models = {}

dummy = DummyClassifier(strategy="most_frequent", random_state=RANDOM_STATE)
dummy.fit(X_train, y_train)
dummy_pred = dummy.predict(X_test)

evaluate_print("Dummy Baseline", y_test, dummy_pred)
results.append({"model":"Dummy", **metrics_dict(y_test, dummy_pred)})
models["Dummy"] = {"model": dummy, "threshold": 0.5}



Dummy Baseline
Accuracy : 0.53725
Precision: 0.0
Recall   : 0.0
F1       : 0.0

Confusion Matrix:
[[10745     0]
 [ 9255     0]]

Classification Report:
              precision    recall  f1-score   support

           0       0.54      1.00      0.70     10745
           1       0.00      0.00      0.00      9255

    accuracy                           0.54     20000
   macro avg       0.27      0.50      0.35     20000
weighted avg       0.29      0.54      0.38     20000



## 3) Logistic Regression (sparse-friendly baseline)

This is a strong, fast baseline for high-dimensional sparse data.


In [4]:
lr = LogisticRegression(
    solver="saga",
    max_iter=5000,
    class_weight="balanced",
    n_jobs=-1,
    random_state=RANDOM_STATE
)
lr.fit(X_train, y_train)

lr_proba = lr.predict_proba(X_test)[:, 1]
t_lr, f1_lr = tune_threshold_f1(y_test, lr_proba)
lr_pred = (lr_proba >= t_lr).astype(int)

evaluate_print(f"Logistic Regression (tuned threshold={t_lr:.2f})", y_test, lr_pred, lr_proba)
results.append({"model":"LogisticRegression", "threshold": t_lr, **metrics_dict(y_test, lr_pred, lr_proba)})
models["LogisticRegression"] = {"model": lr, "threshold": t_lr, "proba": lr_proba}



Logistic Regression (tuned threshold=0.55)
Accuracy : 0.99415
Precision: 0.990446543580936
Recall   : 0.9969746083198271
F1       : 0.9936998546120295
ROC-AUC  : 0.9989184068878292
PR-AUC   : 0.9984853803804993

Confusion Matrix:
[[10656    89]
 [   28  9227]]

Classification Report:
              precision    recall  f1-score   support

           0       1.00      0.99      0.99     10745
           1       0.99      1.00      0.99      9255

    accuracy                           0.99     20000
   macro avg       0.99      0.99      0.99     20000
weighted avg       0.99      0.99      0.99     20000



## 4) Support Vector Machine (applicable): LinearSVC + Calibration

- `LinearSVC` works well with sparse features.
- It does not output probabilities, so we calibrate it to get `predict_proba`.


In [ ]:
svm = LinearSVC(class_weight="balanced", random_state=RANDOM_STATE)
svm_cal = CalibratedClassifierCV(
    estimator=svm,
    method="sigmoid",
    cv=StratifiedKFold(n_splits=3, shuffle=True, random_state=RANDOM_STATE)
)
svm_cal.fit(X_train, y_train)

svm_proba = svm_cal.predict_proba(X_test)[:, 1]
t_svm, f1_svm = tune_threshold_f1(y_test, svm_proba)
svm_pred = (svm_proba >= t_svm).astype(int)

evaluate_print(f"Linear SVM (calibrated, tuned threshold={t_svm:.2f})", y_test, svm_pred, svm_proba)
results.append({"model":"LinearSVM_Calibrated", "threshold": t_svm, **metrics_dict(y_test, svm_pred, svm_proba)})
models["LinearSVM_Calibrated"] = {"model": svm_cal, "threshold": t_svm, "proba": svm_proba}


## 5) Random Forest (with automatic dimensionality reduction if needed)

RandomForest can be memory-heavy with very wide one-hot data.  
We handle this safely:
- If features are already small, we train directly on dense.
- If features are huge, we apply `TruncatedSVD` → dense low-dimensional representation → train RF.


In [ ]:
def dense_size_gb(n_rows, n_cols, dtype_bytes=4):
    return (n_rows * n_cols * dtype_bytes) / (1024**3)

MAX_DENSE_GB = 2.0  # safety cap

rf = None
rf_proba = None
rf_pred = None
rf_threshold = 0.5

try:
    gb = dense_size_gb(X_train.shape[0], X_train.shape[1], dtype_bytes=4)
    print(f"Estimated dense size float32: {gb:.2f} GB")

    if gb <= MAX_DENSE_GB:
        Xtr = X_train.toarray().astype(np.float32)
        Xte = X_test.toarray().astype(np.float32)
        print("Training RF on dense features (no SVD).")
    else:
        print("Using TruncatedSVD to reduce dimensionality for RF...")
        svd = TruncatedSVD(n_components=300, random_state=RANDOM_STATE)
        Xtr = svd.fit_transform(X_train).astype(np.float32)
        Xte = svd.transform(X_test).astype(np.float32)

    rf = RandomForestClassifier(
        n_estimators=500,
        max_depth=None,
        min_samples_split=5,
        min_samples_leaf=2,
        class_weight="balanced_subsample",
        n_jobs=-1,
        random_state=RANDOM_STATE
    )
    rf.fit(Xtr, y_train)

    rf_proba = rf.predict_proba(Xte)[:, 1]
    rf_threshold, rf_bestf1 = tune_threshold_f1(y_test, rf_proba)
    rf_pred = (rf_proba >= rf_threshold).astype(int)

    evaluate_print(f"Random Forest (tuned threshold={rf_threshold:.2f})", y_test, rf_pred, rf_proba)
    results.append({"model":"RandomForest", "threshold": rf_threshold, **metrics_dict(y_test, rf_pred, rf_proba)})
    models["RandomForest"] = {"model": rf, "threshold": rf_threshold, "proba": rf_proba, "uses_svd": gb > MAX_DENSE_GB}
except Exception as e:
    print("RandomForest failed:", e)


## 6) XGBoost (with automatic dimensionality reduction if needed)

XGBoost is usually strong.  
Like RF, it can be memory-heavy with extremely wide one-hot features, so we use the same SVD fallback.


In [ ]:
xgb = None
xgb_proba = None
xgb_pred = None
xgb_threshold = 0.5

try:
    from xgboost import XGBClassifier

    gb = dense_size_gb(X_train.shape[0], X_train.shape[1], dtype_bytes=4)
    print(f"Estimated dense size float32: {gb:.2f} GB")

    if gb <= MAX_DENSE_GB:
        Xtr = X_train.toarray().astype(np.float32)
        Xte = X_test.toarray().astype(np.float32)
        print("Training XGBoost on dense features (no SVD).")
    else:
        print("Using TruncatedSVD to reduce dimensionality for XGBoost...")
        svd = TruncatedSVD(n_components=400, random_state=RANDOM_STATE)
        Xtr = svd.fit_transform(X_train).astype(np.float32)
        Xte = svd.transform(X_test).astype(np.float32)

    xgb = XGBClassifier(
        n_estimators=900,
        max_depth=6,
        learning_rate=0.05,
        subsample=0.8,
        colsample_bytree=0.8,
        reg_alpha=1.0,
        reg_lambda=1.0,
        random_state=RANDOM_STATE,
        n_jobs=-1,
        eval_metric="logloss"
    )
    xgb.fit(Xtr, y_train)

    xgb_proba = xgb.predict_proba(Xte)[:, 1]
    xgb_threshold, xgb_bestf1 = tune_threshold_f1(y_test, xgb_proba)
    xgb_pred = (xgb_proba >= xgb_threshold).astype(int)

    evaluate_print(f"XGBoost (tuned threshold={xgb_threshold:.2f})", y_test, xgb_pred, xgb_proba)
    results.append({"model":"XGBoost", "threshold": xgb_threshold, **metrics_dict(y_test, xgb_pred, xgb_proba)})
    models["XGBoost"] = {"model": xgb, "threshold": xgb_threshold, "proba": xgb_proba, "uses_svd": gb > MAX_DENSE_GB}
except ImportError:
    print("xgboost not installed. Install with: pip install xgboost")
except Exception as e:
    print("XGBoost failed:", e)


## 7) Compare models + select best

We sort by:
1) F1 (primary)
2) PR-AUC (secondary)
3) ROC-AUC (tertiary)


In [ ]:
df_results = pd.DataFrame(results)

# Ensure threshold column exists for all rows
if "threshold" not in df_results.columns:
    df_results["threshold"] = np.nan

df_results_sorted = df_results.sort_values(
    by=["f1", "pr_auc", "roc_auc"],
    ascending=False
).reset_index(drop=True)

print("\n==================== MODEL COMPARISON ====================")
display(df_results_sorted)

best_row = df_results_sorted.iloc[0].to_dict()
best_model_name = best_row["model"]
best_threshold = float(best_row.get("threshold", 0.5))

print("\n🏆 Best model:", best_model_name)
print("Threshold:", best_threshold)
print(best_row)


## 8) Save deployment bundle (best model)

We save:
- best model object
- decision threshold
- preprocessing pipeline (so new client inputs can be transformed consistently)


In [ ]:
bundle = {
    "model_name": best_model_name,
    "model": models[best_model_name]["model"] if best_model_name in models else None,
    "threshold": best_threshold,
    "preprocess_pipeline": artifacts["pipeline"],
    "feature_names": artifacts.get("feature_names", None),
    "metadata": {"random_state": RANDOM_STATE}
}

joblib.dump(bundle, "highrisk_model_bundle.joblib")
print("✅ Saved: highrisk_model_bundle.joblib")


## 9) Quick inference example (client-like record)

This example assumes your preprocessing pipeline expects raw fields used in preprocessing.
If your preprocessing expects other raw columns, adjust the `client` dict keys.


In [ ]:
def predict_highrisk_from_raw(client: dict, bundle_path="highrisk_model_bundle.joblib"):
    b = joblib.load(bundle_path)
    model = b["model"]
    pipe = b["preprocess_pipeline"]
    thr = b["threshold"]

    df_row = pd.DataFrame([client])
    X_row = pipe.transform(df_row)

    # Models here are probabilistic (LR, calibrated SVM, RF, XGB)
    proba = float(model.predict_proba(X_row)[:, 1][0]) if hasattr(model, "predict_proba") else None
    pred = int(proba >= thr) if proba is not None else int(model.predict(X_row)[0])

    return {"high_risk": pred, "probability_high_risk": proba, "threshold": float(thr), "model": b["model_name"]}

example_client = {
    "LATITUDE": 24.8607,
    "LONGITUDE": 67.0011,
    "DATE": "2026-01-04",
    "TIME": "21:30:00"
}

predict_highrisk_from_raw(example_client)
